# 4 — Federating across cohorts, from the command line

Notebook 3 left three cohorts each with their own clusters. This one pools them.

What crosses the boundary is **sufficient statistics** and **bootstrap counts**.
No cohort ships a row. The pooled tree is nonetheless bit-for-bit the tree you
would get by pooling the raw matrices.

Run notebook 3 first: this one reuses the same `sle-run/` directory.

In [ ]:
import os, shutil, subprocess, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Everything is written here, so nothing lands in the repository.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the
    command line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd if isinstance(cmd, list) else [cmd]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
# The SLE data is not in the repository -- download it from Zenodo
# (doi:10.5281/zenodo.20342569) and point SLE at the unpacked folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 100
    abundance = pd.read_csv(SLE / 'abundance.csv').set_index('SampleId')
    meta = pd.read_csv(SLE / 'sample-metadata.csv').set_index('SampleId')
    meta = meta[meta['Included_in_study'] == 'Included']
    abundance = abundance.loc[meta.index]
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
    print(f'SLE data: {abundance.shape[0]} samples x {abundance.shape[1]} reagents')
else:
    # A stand-in with the same shape of problem, so every command below runs
    # unchanged without the download: two batches, a case/control split, and a
    # few proteins measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    rng = np.random.default_rng(0)
    n, p = 75, 30
    ids = [f'S{i:03d}' for i in range(n)]
    drivers = rng.normal(size=(n, 6))
    X = np.exp(rng.normal(3, 1, size=(1, p)) + drivers @ rng.normal(size=(6, p))
               + rng.normal(scale=0.3, size=(n, p)))
    seqs = [f'seq.{1000+j}.{j%7}' for j in range(p)]
    abundance = pd.DataFrame(X, index=pd.Index(ids, name='SampleId'), columns=seqs)
    batch = np.where(np.arange(n) % 3 == 0, 'B', 'A')
    abundance.loc[batch == 'B'] *= 1.6                     # a real batch shift
    meta = pd.DataFrame({
        'DonorId': ids, 'Batch': batch,
        'Group': np.where(rng.random(n) < 0.25, 'HV', 'SLE'),
        'Sex': rng.choice(['F', 'M'], n, p=[0.85, 0.15]),
        'Age_group': rng.choice(['26-30', '31-35', '36-40', '41-45'], n),
        'Disease_activity': rng.choice(['Remission', 'LDA', 'MDA', 'HDA'], n),
        'SLEDAI_2K': rng.integers(0, 14, n)}, index=pd.Index(ids, name='SampleId'))
    # names, with three proteins deliberately measured twice
    gene = [f'G{j:02d}' for j in range(p)]
    for a, b in [(1, 2), (10, 11), (20, 21)]:
        gene[b] = gene[a]
    fm = pd.DataFrame({'SeqId': seqs, 'TargetFullName': gene, 'GeneSymbol': gene})
    dup = fm['GeneSymbol'].duplicated(keep=False)
    fm.loc[dup, 'GeneSymbol'] = fm.loc[dup, 'GeneSymbol'] + '_' + fm.loc[dup, 'SeqId']
    fm.to_csv('feature_metadata.txt', sep='\t', index=False)
    FEATURE_MAP = 'feature_metadata.txt'
    print('SLE data not found -- using a stand-in of the same shape.')
    print(f'stand-in: {abundance.shape[0]} samples x {abundance.shape[1]} reagents')

In [ ]:
# Three cohorts, donors kept whole so repeat timepoints never straddle a boundary.
rng = np.random.default_rng(42)
donors = meta.groupby('DonorId').size().index.to_numpy()
who = dict(zip(rng.permutation(donors), range(len(donors))))
which = meta['DonorId'].map(lambda d: 'ABC'[who[d] % 3])

# SLEDAI banded, so it reads as a strip rather than fifteen shades of one colour.
out = meta.copy()
out['SLEDAI_band'] = pd.cut(pd.to_numeric(out['SLEDAI_2K'], errors='coerce'),
                            [-0.1, 0, 4, 8, 30], labels=['0', '1-4', '5-8', '9+'])
out = out.astype({'SLEDAI_band': str}).replace('nan', 'NA').fillna('NA')
out.to_csv('meta.csv')
abundance.to_csv('cohort_all.csv')
for c in 'ABC':
    abundance.loc[which[which == c].index].to_csv(f'cohort{c}.csv')
print({c: int((which == c).sum()) for c in 'ABC'})

In [ ]:
run(['pvclust-py', 'project-features', '--project', 'all',
     '--matrix', 'cohort_all.csv', '--log2',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--metadata', 'meta.csv', '--top-variable', str(TOPVAR),
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol'])

In [ ]:
# The flags every command shares. Written out in full each time below, so you can
# copy any single cell straight into a terminal.
COMMON = ['--log2', '--adjust', 'combat', '--batch-col', 'Batch',
          '--protect', 'Group', '--metadata', 'meta.csv',
          '--shared-features', 'all_features.csv',
          '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol']
DIST = ['--dist', 'correlation', '--linkage', 'average']
print(' '.join(COMMON))

## What actually leaves each cohort

Four `p x p` matrices: pairwise counts, sums, sums of squares, and the Gram matrix.
Every entry is a sum over rows, which is exactly why they add across cohorts.

**The privacy rule is `p < n`.** The Gram matrix has rank `min(n, p)`, so when a
cohort has fewer samples than objects its row space is fully exposed. At `n = 1` the
Gram is rank one and hands back the row itself (Homer et al., 2008).

In [ ]:
for c in 'ABC':
    run(['pvclust-py', 'project-stats', '--project', f'cohort{c}',
         '--matrix', f'cohort{c}.csv', *COMMON])

## Pass one: the pooled tree and the catalogue

Pooling the statistics gives the exact pooled distance matrix, and from it the
federated tree. Its clusters become the **catalogue** every cohort will count
against.

In [ ]:
STATS = [a for c in 'ABC' for a in ('--stats', f'cohort{c}_stats.npz')]
run(['pvclust-py', 'aggregate-trees', '--labels', 'cohortA_labels.txt',
     *STATS, *DIST])

## Pass two: everyone counts against that one catalogue

This second pass is **not optional**. Pooling per-cohort trees instead finds
clusters no cohort agrees on, because each cohort's tree contains different
clusters and a count only adds to another count when both describe the same
member set. The catalogue is what makes the counts addable.

Clusters are identified by content — a hash of the sorted member names — never by
position in a tree.

In [ ]:
for c in 'ABC':
    run(['pvclust-py', 'count-edges', '--project', f'cohort{c}',
         '--catalogue', 'federated_pvclust_catalogue.csv',
         '--matrix', f'cohort{c}.csv', *COMMON, *DIST, '--n-boot', str(NBOOT)])

## Federated AU

Counts are summed and `msfit` is run once, at the aggregator, on the pooled
bootstrap probabilities. Each cohort's scales are relative to its own `n`, so they
are re-expressed against the pooled `n` before adding; without that rescaling the
curve is fitted against the wrong abscissa and the federated AU collapses.

In [ ]:
COUNTS = [a for c in 'ABC' for a in ('--counts', f'cohort{c}_counts.csv')]
run(['pvclust-py', 'aggregate-trees', '--labels', 'cohortA_labels.txt',
     *STATS, *COUNTS, *DIST, '--alpha', '0.95'])

## The k-means catalogue

The same pooled distance, partitioned flat instead of hierarchically. It uses
**k-medoids**, because Lloyd's algorithm needs centroids — coordinates — while
medoids need only distances, and distances are what federate.

The outputs carry the method in their names, so this does not overwrite the
pvclust catalogue.

In [ ]:
run(['pvclust-py', 'aggregate-trees', '--labels', 'cohortA_labels.txt',
     *STATS, *DIST, '--partition', 'kmeans', '--k', '10'])
print(sorted(p.name for p in Path('.').glob('federated_*')))

## Back to each cohort

There is no single global answer. Each cohort gets **its own** outcome: its local
AU for every federated cluster, next to the federated value. A cluster with high
federated AU and low local AU is one this cohort could not have found alone, which
is the gain from federating made concrete.

`--plot` draws the cohort's own data arranged in the federation's order.

In [ ]:
for c in 'ABC':
    run(['pvclust-py', 'apply-edges', '--project', f'cohort{c}',
         '--matrix', f'cohort{c}.csv', *COMMON, *DIST,
         '--federated-edges', 'federated_pvclust_edges.csv',
         '--n-boot', str(NBOOT), '--alpha', '0.95', '--plot'])

In [ ]:
sup = pd.read_csv('cohortA_from-federated_edge_support.csv')
print('where federating helped cohort A most:')
print(sup.head(8)[['n_members', 'au_local', 'au_federated', 'gain']].to_string(index=False))

## Did the tree find biology or noise?

A protein measured by two reagents is a positive control with a known answer: the
two must land together, or the tree is measuring noise. The permutation null asks
how tight a random group of the same size would be, which turns *they are close*
into a *p*-value.

Not every miss is a failure. On the real data SIGLEC5 merges with its paralog
SIGLEC14, whose extracellular domains are near-identical so the reagents
cross-react, and SAA1 pairs with CRP because both are IL-6-driven acute-phase
reactants. Those are correct answers, not clustering errors.

In [ ]:
import numpy as np
from pvclust_py.core import PvclustResult
from pvclust_py.hclust import linkage, edge_table
from pvclust_py.validate import reagent_groups, cocluster, summary

D = pd.read_csv('federated_distance.csv', index_col=0)
labels = list(D.columns)
au = dict(zip(*pd.read_csv('federated_pvclust_edges.csv')[['edge_id', 'au']].values.T))
Z = linkage(D.to_numpy(float), 'average')
edges = [dict(e, au=float(au.get(e['edge_id'], 0.0)), bp=0.0, si=0.0, se_au=0.0,
              se_bp=0.0, se_si=0.0, v=0.0, c=0.0, df=0, rss=0.0, pchi=1.0)
         for e in edge_table(Z, labels)]
res = PvclustResult(linkage=Z, labels=labels, edges=edges,
                    count=np.zeros((len(edges), 1)), r=np.array([1.0]),
                    nboot=np.array([1]), method_dist='correlation',
                    method_hclust='average', cluster='columns')

ann = pd.read_csv(FEATURE_MAP, sep='\t')
groups = reagent_groups(labels, annotation=ann, name_col='TargetFullName')
if groups:
    print(summary(cocluster(res, groups, alpha=0.95)))
else:
    print('no protein in this feature set is measured by more than one reagent')

---
**What each cohort ends up with.**

| artifact | contents |
|---|---|
| `federated_distance.csv` | the pooled `p x p` distance |
| `federated_pvclust_catalogue.csv` | clusters of the pooled tree |
| `federated_kmeans_catalogue.csv` | the flat k-medoids partition |
| `federated_pvclust_edges.csv` | per cluster: si, au, bp, standard errors, v, c, pchi |
| `<cohort>_from-federated_edge_support.csv` | local vs federated AU, per cluster |
| `<cohort>_from-federated_modules.csv` | the non-overlapping module definitions |
| `<cohort>_from-federated_module_scores.csv` | one score per sample per module |

**One caveat to carry forward.** Clustering patients means resampling analytes, and
analytes are co-expressed rather than independent, so AU on that axis is
anti-conservative. It is a real number, but an optimistic one. Report it as such.